# CEMA_FPGA

## Imports and Parameters

In [1]:
#Imports
import chipwhisperer as cw
import sys
from Crypto.Cipher import AES
from chipwhisperer.common.utils import util
import time
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import trange
from tqdm import tnrange
from scipy.stats import ttest_ind
# from scipy.stats import _moment
import scipy

import sys
sys.path.insert(1,'../../SCApeGoat-main/')

from WPI_SCA_LIBRARY.CWScope import *
from WPI_SCA_LIBRARY.LeakageModels import *
%run "../function/CEMA_functions.ipynb"
%run "../function/AES_model.ipynb"
#Global Variables
N = 1000000
no_samples =10400
clk_spd = 5E6
vcc_in = 1.0

In [ ]:
temp_trace[:,40:].shape

In [ ]:
#variables
K_dev = util.hexStrToByteArray("01 23 45 67 89 ab cd fe 12 34 56 78 9a bc de f0")
# K_dev = bytearray()
K_gen = util.hexStrToByteArray("01 23 45 67 89 ab cd fe 12 34 56 78 9a bc de f0")
I_fixed = util.hexStrToByteArray("da 39 a3 ee 5e 6b 4b 0d 32 55 bf ef 95 60 18 90")
# I_fixed = bytearray([0x9e] * 16); 
I_rand = bytearray([0x00] *16) #+ [0x11] *16 + [0x55] * 16 + [0xaa] *16 + [0xff] * 16 + [0x88] *16


In [ ]:
import cwtvla.ktp
ktp = cwtvla.ktp.FixedVRandomText()


In [ ]:
# key_5k = []
# pt_5k = []
rpt_5k =[]
for i in range(5000):
    _,p = ktp.next_group_B()
    for i in range(10):
#         key_5k.append(k)
        rpt_5k.append(p)

## Setup Scope and target

In [ ]:
# bitfile="delay_test_@.bit"
bitfile = "impl_3.bit"

In [ ]:
scope_cw = CWScope(None, target_type=cw.targets.CW305,fpga_id='100t',force = True,bsfile= bitfile)

In [ ]:
scope_cw.disconnect()

In [ ]:
clk_spd = 5000000

scope_cw.target.vccint_set(vcc_in)
#Configuration of the PLL Clocks
scope_cw.target.pll.pll_enable_set(True) #Enable PLL chip
scope_cw.target.pll.pll_outenable_set(False, 0) # Disable unused PLL0
scope_cw.target.pll.pll_outenable_set(True, 1)  # Enable PLL
scope_cw.target.pll.pll_outenable_set(False, 2) # Disable unused PLL2
scope_cw.scope.adc.samples = no_samples
# run at 10 MHz:
scope_cw.target.pll.pll_outfreq_set(clk_spd, 1)

# 1ms is plenty of idling time
scope_cw.target.clkusbautooff = True
scope_cw.target.clksleeptime = 1

In [ ]:
# # Reset ADC and No. sample
# scope.clock.clkgen_src = "internal"
# scope.clock.clkgen_freq =10000000
# scope.clock.adc_src = "clkgen_x1"
scope_cw.scope.clock.clkgen_src = "extclk"
scope_cw.scope.clock.adc_mul =40
print(scope_cw.scope.clock.clkgen_locked)
# scope.clock.reset_adc() # make sure the DCM ibs locked
# scope.adc.segments = 8
# print(scope.clock.extclk_freq)
print(scope_cw.scope.clock.freq_ctr)
print(scope_cw.scope.clock.adc_freq)

assert scope_cw.scope.clock.adc_locked, "ADC failed to lock"
scope_cw.scope.gain.db = 15

#switching of gain too low error, please enable before running new designs
scope_cw.scope.adc.segment_cycles=0;
print(scope_cw.scope.XADC.status)
scope_cw.scope.adc.lo_gain_errors_disabled = False

In [ ]:
scope_cw.scope.adc_test()

In [ ]:
# # Reset ADC and No. sample

# # scope.clock.clkgen_src = "internal"
# # scope.clock.clkgen_freq =10000000
# # scope.clock.adc_src = "clkgen_x1"
# scope.clock.clkgen_src = "extclk"
# scope.clock.adc_mul = 1
# print(scope.clock.clkgen_locked)
# # scope.clock.reset_adc() # make sure the DCM ibs locked

# # print(scope.clock.extclk_freq)
# print(scope.clock.freq_ctr)
# print(scope.clock.adc_freq)
scope_cw.scope.adc.samples = 450


## Testing 

In [ ]:
#Check Trace
# cipher = AES.new(bytes(K_gen), AES.MODE_ECB)
# I_rand = cipher.encrypt(bytes(I_rand))
traces = []
for i in trange(10):
    trace = cw.capture_trace(scope_cw.scope, scope_cw.target, bytearray(list(keys_pt[i])),bytearray(list(k)),poll_done=True)
    print(f"after capture={scope_cw.scope.adc.trig_count}" )
    traces.append(trace.wave)
#     no_samples = 15
    # print(len(trace.wave))
    # print(I_rand)
#     print(list(trace.textout))

In [ ]:
plt.plot(np.mean(np.array(traces),axis=0))

In [ ]:
plt.plot(traces[0])

## Scapegoat paths

In [ ]:
emsca = FileParent("CEMA",".\\",True)
pt = emsca.get_experiment("pt_keys")
keys_pt = pt.get_dataset("keys").read_data(0,10000)
random_pt = pt.get_dataset("plaintexts").read_data(0,10000)
fixed_pt = pt.get_dataset("fixed_pt").read_data(0,10000)

In [ ]:
#new experimetns and datasets
# ptc_5x10k = emsca.get_experiment("pt_keys_5kx10_rambus")

ptc_5x10k.add_dataset("keys",key_5k,datatype="uint8")
ptc_5x10k.add_dataset("plaintexts",rpt_5k,datatype="uint8")
ptc_5x10k.add_dataset("fixed_pt",pt_5k,datatype="uint8")

In [ ]:
test_30 = emsca.get_experiment("fpga_cema_1_25mm_30k_impl_1")

In [ ]:
len(test_30.get_dataset("random_10_10").read_all())

In [ ]:
emsca.experiments

## EM Setup 

In [2]:
#initialise the motor
A=XYZ()
X,Y,Z,interface=A.XYZ_setup(velocity=10000,acceleration=10000)

Preparing parameters
MotorControl {'motor': 0, 'target_position': 17755355, 'actual_position': 17755355, 'target_velocity': 0, 'actual_velocity': 0}
MotorControl {'motor': 1, 'target_position': -2994706, 'actual_position': -2994706, 'target_velocity': 0, 'actual_velocity': 0}
MotorControl {'motor': 2, 'target_position': -127959, 'actual_position': -1575342159, 'target_velocity': 0, 'actual_velocity': 0}


In [ ]:
# some methods to use for going to starting position, find more functions at https://github.com/analogdevicesinc/PyTrinamic
# Z.get_actual_position() #gives actual position 
X.move_by(stepsize*10) #moves by specific steps

In [ ]:
# #functions for manual testing

# a = X.get_actual_position()
# X.rotate(-100000)
# X.stop()
# X.move_by(1746112)
# b = X.get_actual_position()
# print(a)
# print(b)
# print(a-b)

# # side length of CW-lite processor in stepper steps
# s_len = 1746000 #160000*12

# # number of steps per axis. square of this number is total steps
# n_steps = 2
# N_traces =50
# # size of steps between measurement points
# stepsize = - s_len // (n_steps )
# print('step size =', stepsize)
#(168057, -2005315)

In [5]:
Z.rotate(1000000)

In [6]:
Z.stop()

In [ ]:
x_test = X.get_actual_position()
y_test = Y.get_actual_position()


In [ ]:
Y.move_to(y1)
X.move_to(x1)


In [ ]:
x1==x_test,y1==y_test

In [ ]:
Y.move_by(int(2878730/2))

In [ ]:
# # side length of CW-lite processor in stepper steps
s_len = 2878800 #160000*12

# # number of steps per axis. square of this number is total steps
n_steps = 10

# # size of steps between measurement points
stepsize = - s_len // (n_steps )
print('step size =', stepsize)

## create experiment and add metadata


In [ ]:
#add/ get experiment 

from datetime import datetime

# # Generate a timestamp with date, hour, and minute
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
N_traces =10000
n_steps = 10
mode = "avg"
test = emsca.add_experiment(f"FPGA_CEMA_1_2mm_{N_traces}_{n_steps}x{n_steps}_40x_impl_3_{mode}")
# test_avg = emsca.get_experiment(f"FPGA_CEMA_1_2mm_{N_traces}_5x5_ave10_35x_impl_1_avg")


#add metadata

test.update_metadata("Author","Dev")
test.update_metadata("#traces",str(N_traces))
test.update_metadata("probe", "1.2mm")
test.update_metadata("filtered","yes")
test.update_metadata("grid_size",str(n_steps))
test.update_metadata("dut", "CW305")
test.update_metadata("Side_length", str(s_len))
test.update_metadata("date",str(timestamp))
test.update_metadata("Clean_supply","yes")
test.update_metadata("Capture","cw_husky")
test.update_metadata("Target","AES_unmasked")
test.update_metadata("grid","10x10")
test.update_metadata("sampling rate",scope_cw.scope.clock.adc_mul)
test.update_metadata("bitfile",bitfile)


In [ ]:
# emsca.delete_experiment(f"fpga_cema_1_25mm_1000_5x5_35x_impl_1")
# emsca.delete_experiment(f"FPGA_CEMA_1_2mm_{N_traces}_5x5_ave10_35x_impl_1_avg")
# N_traces = 1000
test_avg = emsca.add_experiment(f"FPGA_CEMA_1_2mm_{N_traces}_{n_steps}x{n_steps}_40x_impl_3_{mode}_reduced")


In [ ]:
ptc_5x10k.name


In [ ]:
keys_pt = ptc_5k.get_dataset("keys").read_data(0,30000)
random_pt = ptc_5k.get_dataset("plaintexts").read_data(0,30000)
fixed_pt = ptc_5k.get_dataset("fixed_pt").read_data(0,30000)

In [ ]:
ptc_5x10k.get_dataset("plaintexts").read_all()
# N_traces

## capture traces

In [ ]:
#starts from the current location traces grids and return to the starting location after capturing the whole grid 
Grid_Tracing_scapegoat(stepsize,stepsize,n_steps,n_steps,X,Y,Z,interface,scope_cw,ptc_5x10k,test,N_traces)

In [ ]:
def test_to_avg(test,test_avg,avg=10,grid_size =11):

    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            rand = test.get_dataset(f"random_{i}_{j}").read_all()
            fix = test.get_dataset(f"fixed_{i}_{j}").read_all()
            
            # Reshape so each group of 10 rows becomes one block
            f = fix.reshape(-1, avg, fix.shape[1]).mean(axis=1)
            r = rand.reshape(-1, avg, rand.shape[1]).mean(axis=1)
            
            test_avg.add_dataset("fixed_" + str(i) + "_" + str(j), f, datatype="float32")
            test_avg.add_dataset("random_" + str(i) + "_" + str(j), r, datatype="float32")
            
            
#     print(data_avg.shape)  # Should be (1000, 20)

In [ ]:
plt.plot(test.get_dataset("fixed_0_0").read_all()[0])
len(test.get_dataset("random_1_0").read_all())
# plt.plot(test.get_dataset("random_0_0").read_all()[0])
# plt.plot(test_avg.get_dataset("random_1_0").read_all()[999])


In [ ]:
test_to_avg(test,test_avg,avg=10,grid_size =n_steps+1)



In [ ]:
test.dataset

In [ ]:
test_avg.name

## Metrics

In [ ]:
#computes TVLA and plots in a heatmap
t = plot_t_statistic_heatmap(test_avg, grid_size=n_steps+1,rotation_n=2)

In [ ]:
#computes CEMA and plots in a heatmap
CEMA,CEMA_max = plot_CEMA_heatmap(test_avg,ptc_5k,1000,grid_size=n_steps,rotation_n=2,lower_b=35)

In [ ]:
SNR,SNR_db,min_n = plot_SNR_min_heatmap(test,ptc_5k,N_traces,grid_size=n_steps,rotation_n=2,MAX=True)

In [ ]:
SNR,SNR_db,min_n = plot_SNR_min_heatmap(test,pt,N_traces,grid_size=n_steps,rotation_n=2,MAX=False)

In [ ]:
test.get_dataset("random_0_0").read_all()

# testing

In [ ]:
def inv_shift_rows_state(state):
    """
    Apply InvShiftRows to a flat 16-byte AES state (column-major).
    Returns a new list of length 16.
    """
    out = [0]*16
    # row 0 → no shift
    out[0]  = state[0]
    out[4]  = state[4]
    out[8]  = state[8]
    out[12] = state[12]
    # row 1 → shift right by 1
    out[1]  = state[13]
    out[5]  = state[1]
    out[9]  = state[5]
    out[13] = state[9]
    # row 2 → shift right by 2
    out[2]  = state[10]
    out[6]  = state[14]
    out[10] = state[2]
    out[14] = state[6]
    # row 3 → shift right by 3 (or left by 1)
    out[3]  = state[7]
    out[7]  = state[11]
    out[11] = state[15]
    out[15] = state[3]
    return out

INV_SBOX = [
    0x52,0x09,0x6A,0xD5,0x30,0x36,0xA5,0x38,0xBF,0x40,0xA3,0x9E,0x81,0xF3,0xD7,0xFB,
    0x7C,0xE3,0x39,0x82,0x9B,0x2F,0xFF,0x87,0x34,0x8E,0x43,0x44,0xC4,0xDE,0xE9,0xCB,
    0x54,0x7B,0x94,0x32,0xA6,0xC2,0x23,0x3D,0xEE,0x4C,0x95,0x0B,0x42,0xFA,0xC3,0x4E,
    0x08,0x2E,0xA1,0x66,0x28,0xD9,0x24,0xB2,0x76,0x5B,0xA2,0x49,0x6D,0x8B,0xD1,0x25,
    0x72,0xF8,0xF6,0x64,0x86,0x68,0x98,0x16,0xD4,0xA4,0x5C,0xCC,0x5D,0x65,0xB6,0x92,
    0x6C,0x70,0x48,0x50,0xFD,0xED,0xB9,0xDA,0x5E,0x15,0x46,0x57,0xA7,0x8D,0x9D,0x84,
    0x90,0xD8,0xAB,0x00,0x8C,0xBC,0xD3,0x0A,0xF7,0xE4,0x58,0x05,0xB8,0xB3,0x45,0x06,
    0xD0,0x2C,0x1E,0x8F,0xCA,0x3F,0x0F,0x02,0xC1,0xAF,0xBD,0x03,0x01,0x13,0x8A,0x6B,
    0x3A,0x91,0x11,0x41,0x4F,0x67,0xDC,0xEA,0x97,0xF2,0xCF,0xCE,0xF0,0xB4,0xE6,0x73,
    0x96,0xAC,0x74,0x22,0xE7,0xAD,0x35,0x85,0xE2,0xF9,0x37,0xE8,0x1C,0x75,0xDF,0x6E,
    0x47,0xF1,0x1A,0x71,0x1D,0x29,0xC5,0x89,0x6F,0xB7,0x62,0x0E,0xAA,0x18,0xBE,0x1B,
    0xFC,0x56,0x3E,0x4B,0xC6,0xD2,0x79,0x20,0x9A,0xDB,0xC0,0xFE,0x78,0xCD,0x5A,0xF4,
    0x1F,0xDD,0xA8,0x33,0x88,0x07,0xC7,0x31,0xB1,0x12,0x10,0x59,0x27,0x80,0xEC,0x5F,
    0x60,0x51,0x7F,0xA9,0x19,0xB5,0x4A,0x0D,0x2D,0xE5,0x7A,0x9F,0x93,0xC9,0x9C,0xEF,
    0xA0,0xE0,0x3B,0x4D,0xAE,0x2A,0xF5,0xB0,0xC8,0xEB,0xBB,0x3C,0x83,0x53,0x99,0x61,
    0x17,0x2B,0x04,0x7E,0xBA,0x77,0xD6,0x26,0xE1,0x69,0x14,0x63,0x55,0x21,0x0C,0x7D
]


In [ ]:
def generate_cipher(pt_exp):
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
    c = AES(keys[0])
    cta = []
    for i in range(len(plaintexts)):
        ct,_,_,_ = c.encrypt(plaintexts[i])
        cta.append(ct)
    
    pt_exp.add_dataset("ciphertext",cta,datatype="uint8")

In [ ]:
generate_cipher(ptc_5k)

In [ ]:
ct = ptc_5k.get_dataset("ciphertext").read_all()
# ptc_5k.delete_dataset("ciphertext")

In [ ]:
print(ct[0])
print(keys_pt[0])
print(fixed_pt[0])
print(random_pt[0])


In [ ]:
c = AES(keys_pt[0])
pok,emon,r1,r2 = c.encrypt(random_pt[0],round_1=6,round_2=7)
k10 = decode_9(c._Ke[10])
print(r1)

In [ ]:
INV_SBOX[inv_shift_rows_state(ct[0]^k10)[0]]

In [ ]:

import numpy as np

rotated = CEMA  # shape (3,4)
k = 2  # np.rot90(orig, 1) rotates CCW 90°
orig = np.rot90(orig, -k)  # shape becomes (4,3)

# Pick a point in rotated, say (2, 1)
x_rot, y_rot = 2, 1
x_orig, y_orig = reverse_coords_ccw(x_rot, y_rot, orig.shape, k)
# Validate: the value at rotated[2,1] should equal orig[x_orig, y_orig]
assert rotated[x_rot, y_rot] == orig[x_orig, y_orig]
print(rotated[x_rot,y_rot])
print("Rotated point:", (x_rot, y_rot))
print("Mapped back to original:", (x_orig, y_orig),
      "value:", orig[x_orig, y_orig])


In [ ]:
import numpy as np

# Assume "grid" is a 2D array (e.g., shape (7,6)) where each element is a numpy array of shape (5000, N)
# We want to concatenate arrays from positions (2,0) to (6,5) row-wise

# Create a list to hold the arrays to concatenate
# arrays_to_concat = []
# keys_concat = []
fpt_concat = []
# Loop through grid positions from (2,0) to (6,5)
for i in range(2, 7):  # rows 2 to 6 inclusive
    for j in range(5):  # columns 0 to 5 inclusive
        x, y = reverse_coords_ccw(i, j, (10,10), 2)
        fpt_concat.append(test.get_dataset(f"fixed_{x}_{y}").read_data(0, 5000))
#         keys_concat.append(pt.get_dataset("keys").read_data(0, 5000))
        
#         pt_concat.append(pt.get_dataset("plaintexts").read_data(0, 5000))
#         fpt_concat.append(pt.get_dataset("fixed_pt").read_data(0, 5000))
        
#     keys = 
#     plaintexts = 
# Concatenate all arrays along axis 0 (row-wise)
# final_array = np.concatenate(arrays_to_concat, axis=0)
# final_karray = np.concatenate(keys_concat, axis=0)
final_fptarray = np.concatenate(fpt_concat, axis=0)

# final_array now has shape (75000, N)
# best_guess, max_correlation = plot_CEMA_traces_temp(final_array[:,35:],final_karray, final_ptarray,len(final_array),target_byte=0,div=2000)

In [ ]:
arr = final_array
if np.isnan(arr).any():
    print("NaN found, replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0)

print(arr)

In [ ]:
len(final_fptarray[0])


In [ ]:
n_clockwise = (k) % 4

rows, cols = orig.shape


print(n_clockwise)
print(cols)
print(cols - 1 - x_rot)

In [ ]:
plt.plot(r[0])

In [ ]:
num = 20000
f,r = scope_cw.capture_traces_tvla(num,keys_pt,fixed_pt,keys_pt,random_pt)

In [ ]:
x=5
y=0
grid_size = (6,6)
n_rot=2


i,j = reverse_coords_ccw(x,y,grid_size,n_rot)
print(i,j)

temp_trace = test.get_dataset(f"random_{i}_{j}").read_all()
m,c = plot_CEMA_traces_temp(temp_trace,ptc_5k,1000,target_byte=0,div = 10,visualize_correct=True,model=4,r1=10,r2=7)

In [ ]:
lol = plot_CEMA_wr(test, pt, num=5000, target_byte=0, x=x, y=y, div=25)

In [ ]:
x,y = reverse_coords_ccw(3,0,(10,10),2)
print(x,y)

In [ ]:
# test.add_dataset("fixed",f,datatype=float)
# # test.add_dataset("random",r,datatype=float)
# correct_key = keys_pt[0][0]
# print(correct_key)

texts = emsca.get_experiment("test")

In [ ]:
texts.add_dataset("fixed_0_",final_fptarray,datatype=float)
texts.add_dataset("random_0_",final_array,datatype=float)

In [ ]:
x=2
y=0
grid_size = (n_steps+1,n_steps+1)
n_rot=2


i,j = reverse_coords_ccw(x,y,grid_size,n_rot)
print(i,j)
t_stat, t_max = test_avg.calculate_t_test(f"fixed_{i}_{j}", f"random_{i}_{j}",visualize=True)


In [ ]:
r_T = test_avg.get_dataset(f"random_{i}_{j}").read_all()

In [ ]:
plt.plot(r_T[190])

# function


In [ ]:
keys = pt_exp.get_dataset("keys").read_data(0, num)
plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
c = AES(keys[0])

In [ ]:


def hd_cpa_last_round(plaintext,c,target_byte=0):
#     c = AES(keys)
    a,b = c.encrypt(plaintext)
    return bin(a[target_byte] ^ b[target_byte]).count('1')

def byte_snr_last_round(plaintext,c,num_traces,target_byte=0):
#     c = AES(keys)

    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        _,leakage[i] = c.encrypt(plaintext[i])
#         print(leakage[i])
    return [row[target_byte] for row in leakage]

In [ ]:
temp_traces = test.get_dataset("random_6_9").read_all()

In [ ]:
# byte_snr_last_round(plaintexts,c,10,target_byte= 0)
plot_SNR_traces(final_array,final_karray,final_ptarray,125000,last_round=False,target_byte=0)
print()

In [ ]:

def plot_SNR_traces(traces, keys,plaintexts, num, target_byte=0, grid_size=5, SNR_type="BYTE",rotation_n = 0,last_round=False):
    """
    Compute and visualize Signal-to-Noise Ratio (SNR) results as a heatmap.

    SNR Types:
    - "BYTE": Uses all possible byte combinations.
    - "FULL": Uses all possible 16-byte key combinations.
    - "HW": Uses the Hamming weight of the byte.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for SNR analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: Type of SNR analysis ("BYTE", "FULL", or "HW").

    Returns:
    - SNR_values_rotated: Rotated array of maximum SNR values.
    - SNR_dB: SNR values in decibels (10 * log10 of SNR_values_rotated).
    """

    # Retrieve keys and plaintext datasets
#     keys = pt_exp.get_dataset("keys").read_data(0, num)
#     plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Select the labeling method based on SNR type
    if SNR_type == "BYTE":
        if last_round:
            c = AES(keys[0])
            labels = byte_snr_last_round(plaintexts,c,num,target_byte=target_byte)
        else:
            labels = no_sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL":
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1]  # Extract the relevant labels for SNR computation
    else:
        print("Incorrect SNR type specified.")
        return -1

    # Initialize array to store SNR values
    SNR_values = np.zeros(num)

    # Get unique label values
    labels_unique = np.unique(labels)

    # Perform SNR analysis across the grid
#     for i in range(num):
#         for j in trange(grid_size):
    sorted_labels = {k: [] for k in labels_unique}  # Dictionary to store traces per label
#             print(f"Processing grid position ({i}, {j})")

    # Retrieve traces for the current grid position
#             traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

    # Organize traces according to their label
    for index, label in enumerate(labels):
        sorted_labels[label].append(np.array(traces[index]))

    # Compute SNR for the given labels and traces
    snr_result = signal_to_noise_ratio(sorted_labels)

    # Store the maximum absolute SNR value
    SNR_values = np.nanmax(np.abs(snr_result))

    # Rotate the heatmap for correct visualization
#     SNR_values_rotated = np.rot90(SNR_values, k=rotation_n )  # Rotate by 90 degrees clockwise

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
#     sns.heatmap(SNR_values_rotated, annot=True, cbar=True, square=True)
    plt.plot(snr_result)
    # Add labels and title based on SNR type
    plt.title(f"SNR ({SNR_type} mode),last round({last_round})")
    plt.xlabel("samples")
    plt.ylabel("SNR")

    # Display the plot
    plt.show()

    # Compute SNR in decibels
    SNR_dB = 10 * np.log10(snr_result)

    return snr_result, SNR_dB


In [ ]:
dick = intermediates_round6(random_pt,keys_pt[0])

In [ ]:
# aes_round6_cpa.py
# Utilities to match aes_core.v signals and run CPA at round 6 (and last round baseline).
from typing import List, Tuple, Dict, Optional
import numpy as np

# ---------- AES primitives (byte-wise) ----------
SBOX = np.array([
    # 256 entries ...
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
], dtype=np.uint8)
INV_SBOX = np.empty_like(SBOX)
INV_SBOX[SBOX] = np.arange(256, dtype=np.uint8)

RCON = np.array([0x00,0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1B,0x36], dtype=np.uint8)

def rot_word(w: np.ndarray) -> np.ndarray:
    return np.roll(w, -1)

def sub_word(w: np.ndarray) -> np.ndarray:
    return SBOX[w]

def xtime(x: np.ndarray) -> np.ndarray:
    return ((x << 1) & 0xFE) ^ ((x >> 7) * 0x1B)

def gm_mul(a: np.ndarray, b: int) -> np.ndarray:
    # multiply vector of bytes a by constant b in GF(2^8)
    if b == 1: return a
    if b == 2: return xtime(a)
    if b == 3: return xtime(a) ^ a
    if b == 9: return xtime(xtime(xtime(a))) ^ a
    if b == 11: return xtime(xtime(xtime(a)) ^ a) ^ a
    if b == 13: return xtime(xtime(xtime(a) ^ a)) ^ a
    if b == 14: return xtime(xtime(xtime(a) ^ a) ^ a)
    raise ValueError("Unsupported mul")

# AES state is 16 bytes in column-major order (like the spec and your RTL):
# indices in a column: [0,4,8,12], [1,5,9,13], [2,6,10,14], [3,7,11,15]
COLS = [[0,4,8,12],[1,5,9,13],[2,6,10,14],[3,7,11,15]]

def shift_rows(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    out[[1,5,9,13]] = state[[5,9,13,1]]
    out[[2,6,10,14]] = state[[10,14,2,6]]
    out[[3,7,11,15]] = state[[15,3,7,11]]
    return out

def inv_shift_rows(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    out[[1,5,9,13]] = state[[13,1,5,9]]
    out[[2,6,10,14]] = state[[10,14,2,6]]  # 2-step rotation is symmetric
    out[[3,7,11,15]] = state[[7,11,15,3]]
    return out

def mix_columns(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    for c in range(4):
        idx = COLS[c]
        s = state[idx]
        out[idx[0]] = (gm_mul(s,2) ^ gm_mul(s[[1]],3) ^ s[[2]] ^ s[[3]])[0]
        out[idx[1]] = (s[[0]] ^ gm_mul(s,2) ^ gm_mul(s[[2]],3) ^ s[[3]])[1-0]  # adjust indexing
        # cleaner version:
        out[idx[1]] = (s[0] ^ gm_mul(s[1:2],2)[0] ^ gm_mul(s[2:3],3)[0] ^ s[3]).astype(np.uint8)
        out[idx[2]] = (s[0] ^ s[1] ^ gm_mul(s[2:3],2)[0] ^ gm_mul(s[3:4],3)[0]).astype(np.uint8)
        out[idx[3]] = (gm_mul(s[0:1],3)[0] ^ s[1] ^ s[2] ^ gm_mul(s[3:4],2)[0]).astype(np.uint8)
    return out

def sub_bytes(state: np.ndarray) -> np.ndarray:
    return SBOX[state]

def add_round_key(state: np.ndarray, rk: np.ndarray) -> np.ndarray:
    return state ^ rk

# ---------- Key expansion (AES-128 → 11 round keys, 16 bytes each) ----------
def expand_key_128(key_bytes: bytes) -> np.ndarray:
    assert len(key_bytes) == 16
    w = np.frombuffer(key_bytes, dtype=np.uint8).copy()
    # 44 words (4-byte), but we’ll output 11 round keys (11*16 bytes)
    rk = np.empty((11,16), dtype=np.uint8)
    rk[0] = w
    temp = w.copy()
    for i in range(1,11):
        t = temp[-4:].copy()
        t = sub_word(rot_word(t))
        t[0] ^= RCON[i]
        block = np.empty(16, dtype=np.uint8)
        block[0:4]   = temp[0:4]   ^ t
        block[4:8]   = temp[4:8]   ^ block[0:4]
        block[8:12]  = temp[8:12]  ^ block[4:8]
        block[12:16] = temp[12:16] ^ block[8:12]
        rk[i] = block
        temp = block
    return rk  # rk[0]=rk0 (master key), rk[10]=rk10

# ---------- Forward simulate to round 6 and extract RTL-like nodes ----------
def intermediates_round6(plaintexts: List[bytes], key: bytes, rond = 6) -> Dict[str, np.ndarray]:
    """
    Returns dict with:
      sbb_o6  : N x 16  (SubBytes outputs at start of round 6)
      shr_o6  : N x 16
      mxc_o6  : N x 16
      state6  : N x 16  (end of round 6 = mxc_o6 XOR rk6)
      rk5, rk6: 16-byte arrays (for reference)
    """
    rk = expand_key_128(key)  # rk0..rk10
    rk5, rk6 = rk[rond-1], rk[rond]

    N = len(plaintexts)
    sbb_o6 = np.zeros((N,16), dtype=np.uint8)
    shr_o6 = np.zeros_like(sbb_o6)
    mxc_o6 = np.zeros_like(sbb_o6)
    state6 = np.zeros_like(sbb_o6)

    for i, P in enumerate(plaintexts):
        state = np.frombuffer(P, dtype=np.uint8).copy()

        # round 0: initial AddRoundKey
        state = add_round_key(state, rk[0])

        # rounds 1..5 (full rounds)
        for rnd in range(1,rond):
            state = sub_bytes(state)
            state = shift_rows(state)
            state = mix_columns(state)
            state = add_round_key(state, rk[rnd])

        # start of round 6: SubBytes on state (this matches sbb_o when round==6)
        sb = sub_bytes(state)       # sbb_o6
        sbb_o6[i] = sb
        sh = shift_rows(sb)         # shr_o6
        shr_o6[i] = sh
        mx = mix_columns(sh)        # mxc_o6
        mxc_o6[i] = mx
        st6 = add_round_key(mx, rk6)  # state_new at end of round 6
        state6[i] = st6

    return {
        "sbb_o6": sbb_o6,
        "shr_o6": shr_o6,
        "mxc_o6": mxc_o6,
        "state6": state6,
        "rk5": rk5.copy(),
        "rk6": rk6.copy()
    }


# ---------- CPA helpers ----------
def hamming_weight(x: np.ndarray) -> np.ndarray:
    return np.unpackbits(x.reshape(-1,1), axis=1).sum(axis=1).astype(np.int16)

def column_indices(col: int) -> List[int]:
    return COLS[col]

def predict_round6_sbox_column_sumHW(plaintexts: List[bytes], guess_rk5_col: bytes, key: bytes, col: int) -> np.ndarray:
    """
    Prediction scalar per trace for CPA target aligned to sbb_o at round 6, one column at a time.
    We simulate rounds 0..5 using the guessed 4 bytes of rk5 for the selected column, and the true key for other columns.
    If you prefer pure-guessing without using the true key elsewhere, set 'key' to any value but also overwrite rk[5] col.
    """
    rk = expand_key_128(key)
    # overwrite the 4 bytes of rk5 for the chosen column with the guess
    idx = column_indices(col)
    rk[5] = rk[5].copy()
    rk[5][idx] = np.frombuffer(guess_rk5_col, dtype=np.uint8)

    preds = np.zeros(len(plaintexts), dtype=np.float64)
    for i, P in enumerate(plaintexts):
        state = np.frombuffer(P, dtype=np.uint8).copy()
        state = add_round_key(state, rk[0])
        for rnd in range(1,6):
            state = sub_bytes(state)
            state = shift_rows(state)
            state = mix_columns(state)
            state = add_round_key(state, rk[rnd])
        sb = sub_bytes(state)  # sbb_o at round 6
        # sum HW of the 4 bytes in this column
        preds[i] = hamming_weight(sb[idx]).sum()
    return preds

def pearson_corr(x: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """
    x: (N,) predictions
    Y: (N, T) traces (windowed)
    returns per-sample correlation (T,)
    """
    x = x.astype(np.float64)
    X = (x - x.mean())
    denom_x = np.sqrt((X*X).sum())
    Yc = Y - Y.mean(axis=0, keepdims=True)
    denom_y = np.sqrt((Yc*Yc).sum(axis=0))
    denom = denom_x * denom_y
    denom[denom == 0] = np.inf
    return (X[:,None] * Yc).sum(axis=0) / denom

# ---------- CPA drivers ----------
def run_cpa_round6_column(plaintexts: List[bytes], traces: np.ndarray, window: Tuple[int,int], key_for_others: bytes, col: int, guess_bytes: List[int]=(0,1,2,3)):
    """
    Column-wise CPA at round 6 using sbb_o6 (sum HW over 4 bytes in the column).
    Brute-forces the selected positions in the column (default: all 4 → 2^32; for faster testing, pass fewer).
    - plaintexts: list of 16-byte plaintexts
    - traces: (N, T) array
    - window: (start, end) sample indices for the round-6 S-box activity
    - key_for_others: a 16-byte key used to compute other columns' round keys (does not need to be true if you guess all 4 bytes)
    - col: which column [0..3]
    - guess_bytes: positions inside the column to brute-force (subset of [0,1,2,3])
    Returns: (best_guess_4bytes, best_corr_value, corr_waveform)
    """
    idx = column_indices(col)
    start, end = window
    Y = traces[:, start:end]

    # prepare a template for the 4-byte guess; unknowns looped, knowns taken from expanded rk5 of key_for_others
    rk = expand_key_128(key_for_others)
    base_col = rk[5][idx].copy()

    # build search space
    positions = list(guess_bytes)
    n_guess = 256 ** len(positions)
    best = (None, -np.inf, None)

    # simple nested loop via np.ndindex
    for vals in np.ndindex(*(256,)*len(positions)):
        guess_col = base_col.copy()
        for p, v in zip(positions, vals):
            guess_col[p] = v
        preds = predict_round6_sbox_column_sumHW(plaintexts, guess_col.tobytes(), key_for_others, col)
        corr = pearson_corr(preds, Y)
        peak = np.max(np.abs(corr))
        if peak > best[1]:
            best = (guess_col.copy(), float(peak), corr.copy())

    return bytes(best[0].tolist()), best[1], best[2]

# ---------- Last-round single-byte baseline (optional) ----------
def run_cpa_last_round_byte(ciphertexts: List[bytes], traces: np.ndarray, window: Tuple[int,int], byte_index_postSR: int, model: str="inv_sbox"):
    """
    Classic last-round CPA per byte.
    model = "inv_sbox" → preds = HW( InvSbox( C[j] ^ k ) )
          = "xor"      → preds = HW( C[j] ^ k )
    Returns: (best_key_byte, best_corr_value, corr_waveform)
    """
    start, end = window
    Y = traces[:, start:end]
    Cj = np.frombuffer(b''.join(ct[byte_index_postSR:byte_index_postSR+1] for ct in ciphertexts), dtype=np.uint8)

    best = (None, -np.inf, None)
    for k in range(256):
        z = Cj ^ k
        if model == "inv_sbox":
            preds = hamming_weight(INV_SBOX[z])
        elif model == "xor":
            preds = hamming_weight(z)
        else:
            raise ValueError("model must be 'inv_sbox' or 'xor'")
        corr = pearson_corr(preds.astype(np.float64), Y)
        peak = np.max(np.abs(corr))
        if peak > best[1]:
            best = (k, float(peak), corr.copy())
    return best


In [ ]:
idx = column_indices(0)
preds = np.zeros(1000, dtype=np.float64)
for i in range(1000):
    
        preds[i] = hamming_weight(dick['sbb_o6'][i,idx]).sum()

In [ ]:
dick
